In [1]:
# NOTE: this requires running the EM stability test in rSLDS_actual
# EM stability test: 1 rSLDS config is run 10x with the same data, 
# to check EM for init and local optima

In [2]:
import os
import numpy as np
import pandas as pd
from math import sqrt
from scipy.stats import t as _t_dist

In [17]:
# --------------------------------------------------------------
# Load files
# --------------------------------------------------------------

def load_files(BASE_DIR, FILENAME_PATTERN, VALUE_COL, IDX, n_files):

    print("\nLoading files")
    
    dfs = {}
    missing = []
    for i in range(1, n_files + 1):
        path = os.path.join(BASE_DIR, FILENAME_PATTERN.format(i))
        if os.path.exists(path):
            print(f"Loading {path}")
            df = pd.read_csv(path)
            df["__file_id__"] = i
            dfs[i] = df
        else:
            missing.append(path)
    
    if missing:
        print("Warning: missing files:")
        for p in missing:
            print("  -", p)
    
    assert len(dfs) > 0, "No input files found. Set BASE_DIR correctly."

    print("\n")
    
    return dfs

In [18]:
# --------------------------------------------------------------
# t-quantile helper
# --------------------------------------------------------------

def _t_quantile(p, df_):
    return _t_dist.ppf(p, df_)

def em_stability_summary(
    dfs,
    cfg,
    n_regimes,
    dim_latent,
    value_col=VALUE_COL,
    alpha=0.05,
):
    """
    Across gridsearch_results1..10:
      - select rows matching (cfg, n_regimes, dim_latent)
      - within each file: clean value_col, take mean over matching rows
      - treat those per-file means as EM runs
    Returns:
      DataFrame with columns:
        config, n_regimes, dim_latent, mean, ci_low, ci_high
    """
    run_vals = []

    for i, df in dfs.items():
        dfi = df[
            (df["config"] == cfg) &
            (df["n_regimes"] == n_regimes) &
            (df["dim_latent"] == dim_latent)
        ]
        if dfi.empty:
            continue

        if value_col not in dfi.columns:
            raise KeyError(f"Column '{value_col}' missing in file {i}.")

        vals = (dfi[value_col]
                .replace([np.inf, -np.inf], np.nan)
                .dropna())
        if len(vals) == 0:
            continue

        run_vals.append(vals.mean())

    run_vals = np.array(run_vals, dtype=float)
    run_vals = run_vals[~np.isnan(run_vals)]
    n = len(run_vals)

    if n == 0:
        raise ValueError("No valid observations for requested config triple across files.")

    mean = run_vals.mean()
    if n > 1:
        std = run_vals.std(ddof=1)
        crit = _t_quantile(1 - alpha / 2, n - 1)
        half_width = crit * std / sqrt(n)
        ci_low = mean - half_width
        ci_high = mean + half_width
    else:
        ci_low = ci_high = mean

    return pd.DataFrame([{
        "config": cfg,
        "n_regimes": n_regimes,
        "dim_latent": dim_latent,
        "mean": mean,
        "ci_low": ci_low,
        "ci_high": ci_high,
    }])


In [19]:
# --------------------------------------------------------------
# Create table: jackknife unrestricted
# --------------------------------------------------------------

BASE_DIR = "/Users/chrismader/Python/SLDS/Out/jackknife unrestr"
FILENAME_PATTERN = "gridsearch_results_jk{}.csv"
VALUE_COL = "cagr_rel_ex_ante"
IDX = ["config", "n_regimes", "dim_latent"]

dfs = load_files(BASE_DIR, FILENAME_PATTERN, VALUE_COL, IDX, n_files=15)

targets = [
    ("[y]", 6, 1),
    ("[g,v]", 6, 2),
]

tbl = pd.concat([
        em_stability_summary(
            dfs,
            cfg=cfg,
            n_regimes=n_reg,
            dim_latent=dim_lat,
            value_col=VALUE_COL,
            alpha=0.05,)
        for (cfg, n_reg, dim_lat) in targets],
    ignore_index=True
).round(3)

tbl


Loading files
Loading file 1
Loading file 2
Loading file 3
Loading file 4
Loading file 5
Loading file 6
Loading file 7
Loading file 8
Loading file 9
Loading file 10
Loading file 11
Loading file 12
Loading file 13
Loading file 14
Loading file 15




,config,n_regimes,dim_latent,mean,ci_low,ci_high
0,[y],6,1,0.067,0.064,0.070
1,"[g,v]",6,2,0.042,0.040,0.043


In [ ]:
# --------------------------------------------------------------
# Create table: jackknife restricted
# --------------------------------------------------------------

BASE_DIR = "/Users/chrismader/Python/SLDS/Out/jackknife restr"
FILENAME_PATTERN = "gridsearch_results_jk{}.csv"
VALUE_COL = "cagr_rel_ex_ante"
IDX = ["config", "n_regimes", "dim_latent"]

dfs = load_files(BASE_DIR, FILENAME_PATTERN, VALUE_COL, IDX, n_files=15)

targets = [
    ("factor2_ff3", 4, 3),
    ("factor2_ff3mom", 4, 3),
]

tbl = pd.concat([
        em_stability_summary(
            dfs,
            cfg=cfg,
            n_regimes=n_reg,
            dim_latent=dim_lat,
            value_col=VALUE_COL,
            alpha=0.05,)
        for (cfg, n_reg, dim_lat) in targets],
    ignore_index=True
).round(3)

tbl

In [14]:
# --------------------------------------------------------------
# Create table: stability y61
# --------------------------------------------------------------

BASE_DIR = "/Users/chrismader/Python/SLDS/Out/stability y61"
FILENAME_PATTERN = "gridsearch_results{}.csv"
VALUE_COL = "cagr_rel_ex_ante"
IDX = ["config", "n_regimes", "dim_latent"]

dfs = load_files(BASE_DIR, FILENAME_PATTERN, VALUE_COL, IDX, n_files=10)

targets = [
    ("[y]", 6, 1),
]

tbl = pd.concat([
        em_stability_summary(
            dfs,
            cfg=cfg,
            n_regimes=n_reg,
            dim_latent=dim_lat,
            value_col=VALUE_COL,
            alpha=0.05,)
        for (cfg, n_reg, dim_lat) in targets],
    ignore_index=True
).round(3)

tbl


Loading files
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results1.csv
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results2.csv
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results3.csv
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results4.csv
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results5.csv
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results6.csv
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results7.csv
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results8.csv
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results9.csv
Loading /Users/chrismader/Python/SLDS/Out/stability y61/gridsearch_results10.csv




,config,n_regimes,dim_latent,mean,ci_low,ci_high
0,[y],6,1,0.065,0.061,0.069


In [16]:
# --------------------------------------------------------------
# Create table: stability gv62
# --------------------------------------------------------------

BASE_DIR = "/Users/chrismader/Python/SLDS/Out/stability gv62"
FILENAME_PATTERN = "gridsearch_results{}.csv"
VALUE_COL = "cagr_rel_ex_ante"
IDX = ["config", "n_regimes", "dim_latent"]

dfs = load_files(BASE_DIR, FILENAME_PATTERN, VALUE_COL, IDX, n_files=10)

targets = [
    ("[g,v]", 6, 2),
]

tbl = pd.concat([
        em_stability_summary(
            dfs,
            cfg=cfg,
            n_regimes=n_reg,
            dim_latent=dim_lat,
            value_col=VALUE_COL,
            alpha=0.05,)
        for (cfg, n_reg, dim_lat) in targets],
    ignore_index=True
).round(3)

tbl


Loading files
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results1.csv
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results2.csv
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results3.csv
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results4.csv
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results5.csv
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results6.csv
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results7.csv
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results8.csv
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results9.csv
Loading /Users/chrismader/Python/SLDS/Out/stability gv62/gridsearch_results10.csv




,config,n_regimes,dim_latent,mean,ci_low,ci_high
0,"[g,v]",6,2,0.043,0.041,0.045
